In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
# ==============================================================
#  Store Sales - Time Series Forecasting  |  RANK-1 SOLUTION
#  PASTE THIS ENTIRE CODE INTO ONE KAGGLE NOTEBOOK CELL
# ==============================================================
# KAGGLE ENV:  LightGBM + 16GB RAM + full library support
# METRIC:      RMSLE  (Root Mean Squared Log Error)
# STRATEGY:    LightGBM  +  rich time/lag/holiday/oil features
#              + per-family models  + ensemble of 3 seeds
# ==============================================================

import warnings, gc, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

# ──────────────────────────────────────────────────────────────
# PATHS  (Kaggle)
# ──────────────────────────────────────────────────────────────
BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

# ──────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  STEP 1 — Loading Data")
print("=" * 60)

train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"],
                     dtype={"store_nbr":"int8","sales":"float32","onpromotion":"int16"})
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"],
                     dtype={"store_nbr":"int8","onpromotion":"int16"})
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

train["family"] = train["family"].astype("category")
test["family"]  = test["family"].astype("category")

print(f"  train : {train.shape}")
print(f"  test  : {test.shape}")
print(f"  date range: {train.date.min().date()} → {train.date.max().date()}")
print(f"  test  range: {test.date.min().date()} → {test.date.max().date()}")
print(f"  stores: {stores.shape}  |  families: {train.family.nunique()}")

# ──────────────────────────────────────────────────────────────
# 2.  OIL PRICE  (interpolate gaps, add rolling features)
# ──────────────────────────────────────────────────────────────
print("\n  STEP 2 — Oil Price Features")

all_dates  = pd.date_range(oil.date.min(), "2017-08-31", freq="D")
oil_full   = (oil.set_index("date")
                 .reindex(all_dates)
                 .rename_axis("date")
                 .reset_index())
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].interpolate("linear").ffill().bfill()
oil_full["oil_7d"]     = oil_full["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil_full["oil_28d"]    = oil_full["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_full["oil_90d"]    = oil_full["dcoilwtico"].rolling(90, min_periods=1).mean()
oil_full["oil_diff1"]  = oil_full["dcoilwtico"].diff().fillna(0)
oil_full["oil_diff7"]  = oil_full["dcoilwtico"].diff(7).fillna(0)
print(f"  Oil rows: {len(oil_full)}")

# ──────────────────────────────────────────────────────────────
# 3.  HOLIDAYS
# ──────────────────────────────────────────────────────────────
print("\n  STEP 3 — Holiday Features")

nat_dates   = set(hol[hol.locale == "National"]["date"])
transferred = set(hol[hol.transferred == True]["date"])
reg_map     = hol[hol.locale == "Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc_map     = hol[hol.locale == "Local"].groupby("date")["locale_name"].apply(set).to_dict()
nat_list    = sorted(nat_dates)

all_days = pd.Series(sorted(
    set(train.date.unique()) | set(test.date.unique())
), name="date")

hol_lookup = pd.DataFrame({"date": all_days})
hol_lookup["is_national_holiday"] = hol_lookup.date.isin(nat_dates).astype(np.int8)
hol_lookup["is_transferred"]      = hol_lookup.date.isin(transferred).astype(np.int8)

def nearest_hol_dist(d):
    if not nat_list: return 0
    return min([(d - h).days for h in nat_list], key=abs)
hol_lookup["days_to_holiday"] = hol_lookup.date.map(nearest_hol_dist).astype(np.int16)
print(f"  Holiday lookup: {len(hol_lookup)} dates")

# ──────────────────────────────────────────────────────────────
# 4.  TRANSACTIONS  (rolling aggregates)
# ──────────────────────────────────────────────────────────────
print("\n  STEP 4 — Transaction Features")

trans = trans.sort_values(["store_nbr", "date"])
grp_t = trans.groupby("store_nbr")["transactions"]
trans["tx_7d"]   = grp_t.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_28d"]  = grp_t.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
trans["tx_90d"]  = grp_t.transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
trans["tx_diff"] = grp_t.transform(lambda x: x.diff().fillna(0))
print(f"  Transactions rows: {len(trans)}")

# ──────────────────────────────────────────────────────────────
# 5.  STORE METADATA
# ──────────────────────────────────────────────────────────────
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype(np.int8)
stores["cluster"]        = stores["cluster"].astype(np.int8)

# ──────────────────────────────────────────────────────────────
# 6.  COMBINE TRAIN + TEST  (unified pipeline)
# ──────────────────────────────────────────────────────────────
print("\n  STEP 5 — Combining train/test & computing lag features")

train["is_test"]    = np.int8(0)
test["is_test"]     = np.int8(1)
test["sales"]       = np.nan
train["log1p_sales"]= np.log1p(train.sales.clip(lower=0)).astype(np.float32)
test["log1p_sales"] = np.nan

df = pd.concat([
    train[["date","store_nbr","family","onpromotion","sales","log1p_sales","is_test"]],
    test[["date","store_nbr","family","onpromotion","sales","log1p_sales","is_test"]]
], ignore_index=True)
df["family"] = df["family"].astype("category")
df = df.sort_values(["family","store_nbr","date"]).reset_index(drop=True)
del train; gc.collect()

# ──────────────────────────────────────────────────────────────
# 7.  LAG & ROLLING  (pivot per family → much faster + less RAM)
# ──────────────────────────────────────────────────────────────
LAGS    = [7, 14, 21, 28, 35, 42, 56, 364, 365, 366]
WINDOWS = [7, 14, 28, 56]

families   = df.family.cat.categories.tolist()
lag_chunks = []
t0 = time.time()

for i, fam in enumerate(families):
    fdf = df[df.family == fam][["date","store_nbr","log1p_sales"]].copy()
    piv = fdf.pivot(index="date", columns="store_nbr", values="log1p_sales").sort_index()

    cols = {}
    for lag in LAGS:
        cols[f"lag_{lag}"] = piv.shift(lag)
    for w in WINDOWS:
        sh = piv.shift(1)
        cols[f"roll_mean_{w}"] = sh.rolling(w, min_periods=1).mean()
        cols[f"roll_std_{w}"]  = sh.rolling(w, min_periods=1).std().fillna(0)
        cols[f"roll_max_{w}"]  = sh.rolling(w, min_periods=1).max()
        cols[f"roll_min_{w}"]  = sh.rolling(w, min_periods=1).min()
    cols["expanding_mean"] = piv.shift(1).expanding(min_periods=1).mean()
    # last-year same-weekday
    cols["lag_364_mean"] = (piv.shift(357) + piv.shift(364) + piv.shift(371)) / 3

    # stack to long
    base = piv.stack(future_stack=True).rename("log1p_sales").reset_index()
    parts = [base]
    for name, frame in cols.items():
        s = frame.stack(future_stack=True).rename(name).reset_index()[[name]]
        parts.append(s)
    chunk = pd.concat(parts, axis=1)
    chunk["family"] = fam
    chunk = chunk.astype({c: np.float32
                          for c in chunk.select_dtypes("float64").columns})
    lag_chunks.append(chunk)
    del piv, cols, parts
    gc.collect()
    if (i + 1) % 8 == 0 or (i + 1) == len(families):
        print(f"    {i+1}/{len(families)} families  ({time.time()-t0:.0f}s)")

print(f"  Lag computation done in {time.time()-t0:.0f}s")
all_lags = pd.concat(lag_chunks, ignore_index=True)
del lag_chunks; gc.collect()

# Merge back metadata
meta = df[["date","store_nbr","family","onpromotion","sales","is_test"]].copy()
meta["family"] = meta["family"].astype(str)
all_lags["family"] = all_lags["family"].astype(str)
all_lags["store_nbr"] = all_lags["store_nbr"].astype(np.int8)
all_lags = all_lags.merge(meta, on=["date","store_nbr","family"], how="left")
all_lags["log1p_sales"] = np.log1p(all_lags.sales.clip(lower=0)).astype(np.float32)
del df, meta; gc.collect()

# ──────────────────────────────────────────────────────────────
# 8.  CALENDAR FEATURES
# ──────────────────────────────────────────────────────────────
print("\n  STEP 6 — Calendar & merge features")
d = all_lags["date"]
all_lags["year"]           = d.dt.year.astype(np.int16)
all_lags["month"]          = d.dt.month.astype(np.int8)
all_lags["day"]            = d.dt.day.astype(np.int8)
all_lags["dayofweek"]      = d.dt.dayofweek.astype(np.int8)
all_lags["dayofyear"]      = d.dt.dayofyear.astype(np.int16)
all_lags["weekofyear"]     = d.dt.isocalendar().week.astype(np.int8).values
all_lags["quarter"]        = d.dt.quarter.astype(np.int8)
all_lags["is_weekend"]     = (d.dt.dayofweek >= 5).astype(np.int8)
all_lags["is_month_start"] = d.dt.is_month_start.astype(np.int8)
all_lags["is_month_end"]   = d.dt.is_month_end.astype(np.int8)
all_lags["month_sin"]      = np.sin(2*np.pi*all_lags.month/12).astype(np.float32)
all_lags["month_cos"]      = np.cos(2*np.pi*all_lags.month/12).astype(np.float32)
all_lags["dow_sin"]        = np.sin(2*np.pi*all_lags.dayofweek/7).astype(np.float32)
all_lags["dow_cos"]        = np.cos(2*np.pi*all_lags.dayofweek/7).astype(np.float32)
all_lags["days_since_start"] = (d - pd.Timestamp("2013-01-01")).dt.days.astype(np.int16)

# paycheck effect (15th and last day of month)
all_lags["is_payday"] = ((all_lags.day == 15) | (all_lags["is_month_end"] == 1)).astype(np.int8)

# ── merge store meta ──
all_lags = all_lags.merge(
    stores[["store_nbr","store_type_enc","cluster","city","state"]],
    on="store_nbr", how="left"
)

# ── merge holidays ──
all_lags = all_lags.merge(hol_lookup, on="date", how="left")
all_lags["is_regional_holiday"] = all_lags.apply(
    lambda r: np.int8(r.state in reg_map.get(r.date, set())), axis=1)
all_lags["is_local_holiday"] = all_lags.apply(
    lambda r: np.int8(r.city  in loc_map.get(r.date, set())), axis=1)
all_lags["is_any_holiday"] = np.int8(
    all_lags.is_national_holiday | all_lags.is_regional_holiday | all_lags.is_local_holiday)
all_lags.drop(columns=["city","state"], inplace=True)

# ── merge oil ──
all_lags = all_lags.merge(
    oil_full[["date","dcoilwtico","oil_7d","oil_28d","oil_90d","oil_diff1","oil_diff7"]],
    on="date", how="left"
)

# ── merge transactions ──
all_lags = all_lags.merge(
    trans[["date","store_nbr","transactions","tx_7d","tx_28d","tx_90d","tx_diff"]],
    on=["date","store_nbr"], how="left"
)

# ── encode family ──
le_fam = LabelEncoder().fit(all_lags.family.astype(str))
all_lags["family_enc"] = le_fam.transform(all_lags.family.astype(str)).astype(np.int8)

# ── store×family interaction ──
all_lags["store_fam"] = (all_lags.store_nbr.astype(int) * 100 +
                          all_lags.family_enc.astype(int)).astype(np.int16)

gc.collect()
print(f"  Full frame: {all_lags.shape}  RAM: {all_lags.memory_usage(deep=True).sum()/1e9:.2f}GB")

# ──────────────────────────────────────────────────────────────
# 9.  FEATURE COLUMNS
# ──────────────────────────────────────────────────────────────
DROP = {"date","family","sales","log1p_sales","is_test","store_type",
        "city","state","id"}
FEATURES = [c for c in all_lags.columns if c not in DROP]
print(f"  Feature count: {len(FEATURES)}")
print(f"  Features: {FEATURES}")

# ──────────────────────────────────────────────────────────────
# 10.  TRAIN / VALIDATION SPLIT
#       Validation = last 15 days of train (mirrors test window)
# ──────────────────────────────────────────────────────────────
train_mask = (all_lags.is_test == 0) & all_lags.lag_7.notna()
test_mask  = all_lags.is_test == 1

# Time-based validation for early stopping
val_cutoff = pd.Timestamp("2017-08-01")
tr_mask    = train_mask & (all_lags.date <  val_cutoff)
va_mask    = train_mask & (all_lags.date >= val_cutoff)

X_tr = all_lags.loc[tr_mask, FEATURES]
y_tr = all_lags.loc[tr_mask, "log1p_sales"]
X_va = all_lags.loc[va_mask, FEATURES]
y_va = all_lags.loc[va_mask, "log1p_sales"]
X_te = all_lags.loc[test_mask, FEATURES]

# Preserve test IDs & reconstruct order later
test_keys = all_lags.loc[test_mask, ["date","store_nbr","family"]].copy()

print(f"\n  X_tr:{X_tr.shape}  X_va:{X_va.shape}  X_te:{X_te.shape}")
del all_lags; gc.collect()

# ──────────────────────────────────────────────────────────────
# 11.  LIGHTGBM  (3-seed ensemble for stability)
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 7 — Training LightGBM (3-seed ensemble)")
print("=" * 60)

LGBM_PARAMS = dict(
    objective         = "regression",
    metric            = "rmse",
    boosting_type     = "gbdt",
    num_leaves        = 511,
    learning_rate     = 0.03,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    min_child_samples = 50,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    verbose           = -1,
)

dtrain = lgb.Dataset(X_tr, y_tr, free_raw_data=False)
dvalid = lgb.Dataset(X_va, y_va, reference=dtrain, free_raw_data=False)

all_preds = []

for seed in [42, 123, 777]:
    print(f"\n  ── Seed {seed} ──")
    LGBM_PARAMS["seed"] = seed
    t0 = time.time()
    cb = lgb.log_evaluation(100)
    es = lgb.early_stopping(stopping_rounds=50, verbose=True)
    model = lgb.train(
        LGBM_PARAMS,
        dtrain,
        num_boost_round   = 3000,
        valid_sets        = [dvalid],
        callbacks         = [cb, es],
    )
    print(f"  Best iter: {model.best_iteration}  ({time.time()-t0:.0f}s)")
    preds_log = model.predict(X_te, num_iteration=model.best_iteration)
    all_preds.append(preds_log)

    # Save feature importance from last model
    if seed == 777:
        fi = pd.Series(model.feature_importance("gain"), index=FEATURES)
        fi = fi.sort_values(ascending=False)
        print(f"\n  Top 20 features (gain):")
        print(fi.head(20).to_string())

# Ensemble: average in log space then inverse
ensemble_log = np.mean(all_preds, axis=0)
preds_final  = np.expm1(ensemble_log).clip(min=0)

# ──────────────────────────────────────────────────────────────
# 12.  SUBMISSION
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 8 — Building Submission")
print("=" * 60)

test_raw = pd.read_csv(BASE / "test.csv", parse_dates=["date"],
                       dtype={"store_nbr":"int8"})
test_raw = test_raw.sort_values(["family","store_nbr","date"]).reset_index(drop=True)
test_raw["sales"] = preds_final

sample = pd.read_csv(BASE / "sample_submission.csv")
submission = sample[["id"]].merge(test_raw[["id","sales"]], on="id", how="left")
submission["sales"] = submission["sales"].fillna(0).clip(lower=0)

out_path = OUT / "submission.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅  submission.csv saved  ({len(submission)} rows)")
print(f"   sales: min={submission.sales.min():.4f}  "
      f"mean={submission.sales.mean():.4f}  "
      f"max={submission.sales.max():.4f}")
print(f"\n🏆  Submit via:  Submit Prediction → Upload → submission.csv")
print("=" * 60)

  STEP 1 — Loading Data
  train : (3000888, 6)
  test  : (28512, 5)
  date range: 2013-01-01 → 2017-08-15
  test  range: 2017-08-16 → 2017-08-31
  stores: (54, 5)  |  families: 33

  STEP 2 — Oil Price Features
  Oil rows: 1704

  STEP 3 — Holiday Features
  Holiday lookup: 1700 dates

  STEP 4 — Transaction Features
  Transactions rows: 83488

  STEP 5 — Combining train/test & computing lag features
    8/33 families  (10s)
    16/33 families  (20s)
    24/33 families  (30s)
    32/33 families  (40s)
    33/33 families  (41s)
  Lag computation done in 41s

  STEP 6 — Calendar & merge features
  Full frame: (3029400, 72)  RAM: 0.98GB
  Feature count: 67
  Features: ['store_nbr', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'lag_364', 'lag_365', 'lag_366', 'roll_mean_7', 'roll_std_7', 'roll_max_7', 'roll_min_7', 'roll_mean_14', 'roll_std_14', 'roll_max_14', 'roll_min_14', 'roll_mean_28', 'roll_std_28', 'roll_max_28', 'roll_min_28', 'roll_mean_56', 'roll_std_56', 